## Bond valuation and scenario P&L

Completed on 27 November 2026 as guided practice with synthetic data.

For a bond with payment amounts $C_i$ at times $t_i$ in years, the model price is the sum of the discounted cash flows.

$$
P = \sum_i C_i e^{-r(t_i)t_i},
$$

where $r(t_i)$ is the continuously compounded zero rate for that payment date, expressed as a decimal. The notebook linearly interpolates the quoted rates and shocks at each payment date. By linearity, adding the interpolated shock gives the same rate as shocking the quotes first and then interpolating.

For a position of $q$ signed units, the scenario P&L is $q(P_{\mathrm{shocked}} - P_{\mathrm{base}})$.

### Optional shock input

`bond_prices_calculator` defaults to `shock_basis=0`. In the base case, pandas adds that scalar zero to every rate in the Series. In the shocked case, the function receives a Series in basis points, and pandas matches each shock to its rate by cash-flow index. Both cases use `(zero_rates + shock_basis / 100) / 100`, so the function needs a single rate calculation for bonds with any number of payments. The first division converts basis points to percentage points, and the second converts the adjusted percentage quote to a decimal rate.

### Largest loss contributor

Under the specified scenario, **B03 contributes the largest loss (€26.62)** in the independent repricing. The zero rate at 2.5 years falls from 2.875% to 2.650%, raising the present value of B03’s largest payment and driving up its bond price. The book is short 50 units of B03, so the price increase produces a loss.

### Verification

B01 pays €100 in one year. Its base and shocked prices agree with the independent benchmarks $100e^{-0.025}$ and $100e^{-0.027}$. A separate rerun with the curve shocks set to zero produced zero P&L for each position and for the book.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd


project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
data_dir = project_root / "data" / "05_bond_book_yield_curve_twist"
if not data_dir.is_dir():
    raise FileNotFoundError("Open this notebook from the repository root or notebooks directory.")

positions = pd.read_csv(data_dir / "positions.csv")
curve = pd.read_csv(data_dir / "curve.csv")
cashflows = pd.read_csv(data_dir / "cashflows.csv")

print(positions)
print(curve)
print(cashflows)


  bond_id  signed_units
0     B03           -50
1     B01            80
2     B02          -120
   maturity_years  zero_rate_pct  shock_bps
0             0.5           2.30         30
1             1.0           2.50         20
2             2.0           2.75        -10
3             3.0           3.00        -35
  bond_id  payment_years  cashflow_eur
0     B01            1.0         100.0
1     B02            0.5           2.0
2     B02            1.0           2.0
3     B02            1.5         102.0
4     B03            0.5           2.5
5     B03            1.0           2.5
6     B03            1.5           2.5
7     B03            2.0           2.5
8     B03            2.5         102.5


In [2]:
def bond_prices_calculator(coupons, zero_rates, payment_years_sorted, shock_basis=0):
    adjusted_rates = (zero_rates + shock_basis / 100) / 100
    P = sum(coupons.iloc[i] * np.exp((- adjusted_rates.iloc[i] * payment_years_sorted.iloc[i])) for i in range(len(payment_years_sorted)))
    return P


def linear_inter(lower_time, lower_value, upper_time, upper_value, middle_time):
    rate = (np.abs(middle_time - lower_time) / (upper_time - lower_time)) * upper_value + (np.abs(upper_time - middle_time) / (upper_time - lower_time)) * lower_value
    return rate


def bond_prices(cashflows, curve):

            
    # Renames the column `maturity_years` to `payment_years` so curve can be joined to cash flows
    curve_rename = curve.rename(columns={"maturity_years": "payment_years"})

    cashflows_new = cashflows.merge(curve_rename,
                                how="left",
                                on="payment_years",
                                validate="many_to_one")



    # Data missing from cash-flows, we identify which payment years are not present in `curve` and store it in `missing_years` by using `.isna()` mask
    bond_payment_years_not_in_curve = cashflows_new[cashflows_new["zero_rate_pct"].isna()].reset_index(drop=True)

    missing_years = bond_payment_years_not_in_curve["payment_years"].drop_duplicates()


    # Calculates the distance between each `missing_years` and all the `payment_years`
    distances = [
        curve["maturity_years"].sub(year).abs() for year in missing_years
    ]



    # Checks if any payment years are outside the bounds of the curve data
    years_check = (missing_years > np.max(curve["maturity_years"])) | (missing_years < np.min(curve["maturity_years"]))
    if years_check.any():
        raise ValueError("Payment years are out of bound with curve data")

    # Create an empty list for our dictionaries
    inter_entries = []
    for year, distances_for_year in zip(missing_years, distances):

        # Creates first boolean condition as a mask to filter all the years above and below our `missing_years`
        above = year < curve["maturity_years"]
        below = year > curve["maturity_years"]

        # Given the above mask we filter the values that have the smallest `distances` in their maturity
        row_above = curve.loc[distances_for_year[above].idxmin()]
        row_below = curve.loc[distances_for_year[below].idxmin()]

        # Calculate interpolated values and concat
        zero_rate_inter = linear_inter(row_below["maturity_years"], row_below["zero_rate_pct"], row_above["maturity_years"], row_above["zero_rate_pct"], year)
        shock_bps_inter = linear_inter(row_below["maturity_years"], row_below["shock_bps"], row_above["maturity_years"], row_above["shock_bps"], year)

        new_rows = {"payment_years": year, "new_zero_rate_pct": zero_rate_inter, "new_shock_bps": shock_bps_inter}
        inter_entries.append(new_rows)

    new_rate_n_shock = pd.DataFrame(inter_entries, columns=["payment_years", "new_zero_rate_pct", "new_shock_bps"])


    # Merge everything and fill in the missing values & discard unecessary collumns
    cashflows_complete = cashflows_new.merge(new_rate_n_shock,
                                        how="left",
                                        on="payment_years")


    cashflows_complete["zero_rate_pct"] = cashflows_complete["zero_rate_pct"].fillna(cashflows_complete["new_zero_rate_pct"])
    cashflows_complete["shock_bps"] = cashflows_complete["shock_bps"].fillna(cashflows_complete["new_shock_bps"])

    cashflows_complete = cashflows_complete.iloc[:, 0:5]


    # Creates empty list to populate with dictionaries for each of `bond_id` and their respective `price`
    rows_prices = []
    for bond_id, group in cashflows_complete.groupby("bond_id"):
        row = {"bond_id": bond_id, 
            "price": bond_prices_calculator(
                group["cashflow_eur"],
                group["zero_rate_pct"],  
                group["payment_years"])}
        rows_prices.append(row)

    # Finally creates a dataframe with bonds and their prices
    bond_prices_df = pd.DataFrame(rows_prices)

    
    # Creates empty list to populate with dictionaries for each of `bond_id` and their respective `shock_prices`
    rows_shock_prices = []
    for bond_id, group in cashflows_complete.groupby("bond_id"):
        row = {"bond_id": bond_id, 
            "shock_price": bond_prices_calculator(
                group["cashflow_eur"],
                group["zero_rate_pct"],
                group["payment_years"],
                group["shock_bps"])}
        rows_shock_prices.append(row)
    bond_shock_prices_df = pd.DataFrame(rows_shock_prices)

    bond_price_n_shock = bond_prices_df.merge(bond_shock_prices_df,
                                        how="left",
                                        on="bond_id",
                                        validate="one_to_one")


    # Test
    test_shock = pd.Series(0.0, index=cashflows_complete.index, name="test_shock")
    cashflows_complete["zero_shock"] = test_shock
    test_prices = []
    for bond_id, group in cashflows_complete.groupby("bond_id"):
        row = {"bond_id": bond_id, 
            "price": bond_prices_calculator(
                group["cashflow_eur"],
                group["zero_rate_pct"] + group["zero_shock"],  
                group["payment_years"])}
        test_prices.append(row)
    lookup_test = {row["bond_id"]: row["price"] for row in test_prices}
    lookup_prices = {row["bond_id"]: row["price"] for row in rows_prices}
    differences = {bond_id: lookup_test[bond_id] - lookup_prices[bond_id] for bond_id in lookup_test}
    bond_id_check = all(lookup_test[bond_id] == lookup_prices[bond_id] for bond_id in lookup_prices)
    test = all(differences[bond_id] == 0 for bond_id in differences)
    assert test and bond_id_check
                                        



    return bond_price_n_shock

In [3]:
# Finds shock prices for 
bond_shock_prices_n_positions = (bond_prices(cashflows, curve).merge(positions,
                                        how="left",
                                        on="bond_id",
                                        validate="one_to_one")
                                        .assign(PnL=lambda table: (table["shock_price"] - table["price"]) * table["signed_units"])
                                        )


# Switches collumn order
bond_shock_prices_n_positions = bond_shock_prices_n_positions[["bond_id", "signed_units", "price", "shock_price", "PnL"]]


print(bond_shock_prices_n_positions)
print(f"The total PnL is {bond_shock_prices_n_positions["PnL"].sum():.3f} euros.")

  bond_id  signed_units       price  shock_price        PnL
0     B01            80   97.530991    97.336124 -15.589364
1     B02          -120  101.989544   101.909164   9.645550
2     B03           -50  105.070718   105.603165 -26.622373
The total PnL is -32.566 euros.
